# 06b — Learned visual–graph alignment

This notebook exercises the Phase-6 frozen-encoder projection model on synthetic fixture embeddings. It demonstrates the training contract, not a heritage result.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir() and (ROOT.parent / 'src').is_dir():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
from caypollard.alignment import AlignmentConfig, embedding_collapse_diagnostics, train_joint_alignment
from caypollard.embeddings.store import EmbeddingTable, l2_normalize


## Synthetic aligned modalities
The hidden latent structure is shared, but each modality observes a different linear transformation. Test rows are projected after training but never used for optimisation or checkpoint selection.


In [ ]:
rng = np.random.default_rng(12)
ids = tuple(f'item-{i:02d}' for i in range(18))
latent = l2_normalize(rng.normal(size=(18, 5)))
visual = EmbeddingTable(ids, l2_normalize(latent @ rng.normal(size=(5, 9)) + 0.02*rng.normal(size=(18,9))), {'method':'fixture-visual'})
graph = EmbeddingTable(ids, l2_normalize(latent @ rng.normal(size=(5, 7)) + 0.02*rng.normal(size=(18,7))), {'method':'fixture-G1-context'})
train_ids, validation_ids, test_ids = ids[:12], ids[12:15], ids[15:]


In [ ]:
config = AlignmentConfig(projection_dim=6, epochs=40, patience=8, batch_size=6, learning_rate=0.02, seed=42)
run = train_joint_alignment(visual, graph, train_ids=train_ids, validation_ids=validation_ids, output_ids=ids, config=config)
visual_shared, graph_shared, joint_shared = run.as_tables()
{'best_epoch': run.metadata['best_epoch'], 'validation_loss': run.metadata['best_validation_loss'], 'test_optimisation': run.metadata['test_optimisation']}


In [ ]:
embedding_collapse_diagnostics(joint_shared)


## Interpretation gate
A successful fixture run only verifies software behaviour. The learned model becomes scientific evidence only after comparison with visual-only, valid G1/G2 graph-only, late-fusion, and reranking baselines under `protocol-v0.4`. A G0 Iconclass-oracle input remains circular regardless of how elegant the loss curve looks.
